<a href="https://colab.research.google.com/github/bumsootead/Seoul_Subway_Analytics/blob/main/Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Data Cleaning and Standardization

## Objective
Create a common station-hour ridership layer from official Seoul Metro data.

## Source coverage
- 2024: Full-year total ridership
- 2025: Jul–Dec passenger-type ridership, aggregated to total ridership

## Output datasets
- station_hourly_total.parquet
- station_daily_wide.csv
- station_hourly_passenger_type_2025.parquet
- dim_station.csv

# 2024 file: already contains total passenger counts by date, line, station, direction, and hour.

# 2025 file: contains counts split by passenger type



In [23]:
RAW_2024 = Path(
    r"C:\Users\ryanj\Downloads\서울교통공사_역별 일별 시간대별 승하차인원 정보_20241231.csv"
)

RAW_2025 = Path(
    r"C:\Users\ryanj\Downloads\서울교통공사_1_8호선 역별 일별 시간대별 승객유형별 승하차인원_20251231.csv"
)

In [24]:
df_2024_raw = pd.read_csv(RAW_2024, encoding="cp949")
df_2025_raw = pd.read_csv(RAW_2025, encoding="cp949")

In [25]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
TABLEAU_DIR = PROJECT_ROOT / "data" / "tableau"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
TABLEAU_DIR.mkdir(parents=True, exist_ok=True)

RAW_2024 = RAW_DIR / "서울교통공사_역별 일별 시간대별 승하차인원 정보_20241231.csv"
RAW_2025 = RAW_DIR / "서울교통공사_1_8호선 역별 일별 시간대별 승객유형별 승하차인원_20251231.csv"

In [28]:
print("2024 shape:", df_2024_raw.shape)
print("2025 shape:", df_2025_raw.shape)

display(df_2024_raw.head())
display(df_2025_raw.head())

print(df_2024_raw.columns.tolist())
print(df_2025_raw.columns.tolist())

2024 shape: (199424, 26)
2025 shape: (797226, 27)


,연번,수송일자,호선,역번호,역명,승하차구분,06시이전,06-07시간대,07-08시간대,08-09시간대,...,15-16시간대,16-17시간대,17-18시간대,18-19시간대,19-20시간대,20-21시간대,21-22시간대,22-23시간대,23-24시간대,24시이후
0,1,2024-01-01,1호선,150,서울역,승차,383,257,308,975,...,2716,2882,2871,2685,2922,2031,2279,1729,868,43
1,2,2024-01-01,1호선,150,서울역,하차,249,867,834,1201,...,2615,2501,2829,2095,1833,1465,1031,585,298,82
2,3,2024-01-01,1호선,151,시청,승차,188,92,167,245,...,935,1003,978,1116,951,909,723,462,176,3
3,4,2024-01-01,1호선,151,시청,하차,103,276,292,451,...,834,697,670,564,351,302,272,120,89,38
4,5,2024-01-01,1호선,152,종각,승차,970,374,193,205,...,1383,1578,1435,1422,1287,1318,1106,775,274,8


,연번,수송일자,호선명,역번호,역명,승하차구분,승객유형,06시간대이전,06-07시간대,07-08시간대,...,15-16시간대,16-17시간대,17-18시간대,18-19시간대,19-20시간대,20-21시간대,21-22시간대,22-23시간대,23-24시간대,24시간대이후
0,1,2025-07-01,1호선,150,서울역,승차,일반,272,939,3588,...,2846,3286,6799,9375,4004,2784,2790,1640,807,115
1,2,2025-07-01,1호선,150,서울역,승차,어린이,0,0,2,...,21,21,22,4,10,6,5,7,1,0
2,3,2025-07-01,1호선,150,서울역,승차,중고생,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,2025-07-01,1호선,150,서울역,승차,청소년,3,11,56,...,57,85,39,47,29,28,50,21,17,1
4,5,2025-07-01,1호선,150,서울역,승차,우대권,98,188,331,...,895,744,848,533,367,267,235,126,46,7


['연번', '수송일자', '호선', '역번호', '역명', '승하차구분', '06시이전', '06-07시간대', '07-08시간대', '08-09시간대', '09-10시간대', '10-11시간대', '11-12시간대', '12-13시간대', '13-14시간대', '14-15시간대', '15-16시간대', '16-17시간대', '17-18시간대', '18-19시간대', '19-20시간대', '20-21시간대', '21-22시간대', '22-23시간대', '23-24시간대', '24시이후']
['연번', '수송일자', '호선명', '역번호', '역명', '승하차구분', '승객유형', '06시간대이전', '06-07시간대', '07-08시간대', '08-09시간대', '09-10시간대', '10-11시간대', '11-12시간대', '12-13시간대', '13-14시간대', '14-15시간대', '15-16시간대', '16-17시간대', '17-18시간대', '18-19시간대', '19-20시간대', '20-21시간대', '21-22시간대', '22-23시간대', '23-24시간대', '24시간대이후']


In [29]:
def raw_data_audit(df, date_col, line_col, station_col, direction_col):
    return {
        "rows": len(df),
        "columns": len(df.columns),
        "date_min": df[date_col].min(),
        "date_max": df[date_col].max(),
        "null_cells": int(df.isna().sum().sum()),
        "line_station_pairs": df.groupby([line_col, station_col]).ngroups,
        "directions": df[direction_col].value_counts().to_dict()
    }

print(raw_data_audit(
    df_2024_raw, "수송일자", "호선", "역명", "승하차구분"
))

print(raw_data_audit(
    df_2025_raw, "수송일자", "호선명", "역명", "승하차구분"
))

{'rows': 199424, 'columns': 26, 'date_min': '2024-01-01', 'date_max': '2024-12-31', 'null_cells': 0, 'line_station_pairs': 274, 'directions': {'승차': 99712, '하차': 99712}}
{'rows': 797226, 'columns': 27, 'date_min': '2025-07-01', 'date_max': '2025-12-31', 'null_cells': 0, 'line_station_pairs': 273, 'directions': {'승차': 398613, '하차': 398613}}


### Data-quality note

The 2024 source is already aggregated across passenger types.
The 2025 source is at passenger-type detail and must be aggregated before
it is combined with 2024 total ridership.

No conclusions are drawn until row counts, date coverage, missing values,
and directional values are verified.

Define common names for hourly fields

In [30]:
DIRECTION_MAP = {
    "승차": "board",
    "하차": "alight"
}

KOREAN_HOUR_MAP = {
    # 2025 naming
    "06시간대이전": "before_06",
    "24시간대이후": "after_24",

    # 2024 naming
    "06시이전": "before_06",
    "24시이후": "after_24",

    # Shared naming
    "06-07시간대": "06_07",
    "07-08시간대": "07_08",
    "08-09시간대": "08_09",
    "09-10시간대": "09_10",
    "10-11시간대": "10_11",
    "11-12시간대": "11_12",
    "12-13시간대": "12_13",
    "13-14시간대": "13_14",
    "14-15시간대": "14_15",
    "15-16시간대": "15_16",
    "16-17시간대": "16_17",
    "17-18시간대": "17_18",
    "18-19시간대": "18_19",
    "19-20시간대": "19_20",
    "20-21시간대": "20_21",
    "21-22시간대": "21_22",
    "22-23시간대": "22_23",
    "23-24시간대": "23_24"
}

HOUR_BUCKETS = [
    "before_06", "06_07", "07_08", "08_09", "09_10",
    "10_11", "11_12", "12_13", "13_14", "14_15",
    "15_16", "16_17", "17_18", "18_19", "19_20",
    "20_21", "21_22", "22_23", "23_24", "after_24"
]

In [32]:
def clean_to_hourly_long():
    assert hourly_long["service_date"].notna().all()
    assert hourly_long["direction"].isin(["board", "alight"]).all()
    assert hourly_long["passenger_count"].notna().all()
    assert (hourly_long["passenger_count"] >= 0).all()
    assert hourly_long["hour_bucket"].notna().all()

In [33]:
def clean_to_hourly_long(df, source_year, has_passenger_type):
    df = df.copy()

    # Standardize identifier names.
    df = df.rename(columns={
        "수송일자": "service_date",
        "호선명": "line_name",
        "호선": "line_name",
        "역번호": "station_code",
        "역명": "station_name",
        "승하차구분": "direction",
        "승객유형": "passenger_type"
    })

    # Supplier row number is not analytically useful.
    df = df.drop(columns=["연번"], errors="ignore")

    # Clean dimensions.
    df["service_date"] = pd.to_datetime(df["service_date"], errors="coerce")
    df["line_name"] = df["line_name"].astype(str).str.strip()
    df["station_code"] = df["station_code"].astype(str).str.strip()
    df["station_name"] = df["station_name"].astype(str).str.strip()
    df["direction"] = df["direction"].map(DIRECTION_MAP)

    # 2024 has no passenger-type detail.
    if has_passenger_type:
        df["passenger_type"] = df["passenger_type"].astype(str).str.strip()
    else:
        df["passenger_type"] = "all_passengers"

    # Identify and clean hour columns.
    hour_columns = [column for column in df.columns if column in KOREAN_HOUR_MAP]

    for column in hour_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0)
        df[column] = df[column].astype("int64")

    # Convert 20 separate hour columns into rows.
    hourly_long = df.melt(
        id_vars=[
            "service_date",
            "line_name",
            "station_code",
            "station_name",
            "direction",
            "passenger_type"
        ],
        value_vars=hour_columns,
        var_name="raw_hour_bucket",
        value_name="passenger_count"
    )

    hourly_long["hour_bucket"] = hourly_long["raw_hour_bucket"].map(
        KOREAN_HOUR_MAP
    )

    # Add calendar fields for analysis and Tableau.
    hourly_long["source_year"] = source_year
    hourly_long["day_of_week"] = hourly_long["service_date"].dt.day_name()
    hourly_long["is_weekday"] = hourly_long["service_date"].dt.dayofweek < 5
    hourly_long["is_weekend"] = ~hourly_long["is_weekday"]
    hourly_long["month"] = hourly_long["service_date"].dt.month
    hourly_long["year_month"] = (
        hourly_long["service_date"].dt.to_period("M").astype(str)
    )

    return hourly_long[
        [
            "service_date",
            "source_year",
            "line_name",
            "station_code",
            "station_name",
            "direction",
            "passenger_type",
            "hour_bucket",
            "passenger_count",
            "day_of_week",
            "is_weekday",
            "is_weekend",
            "month",
            "year_month"
        ]
    ]

# Create standardized datasets 

In [35]:
station_hourly_2024 = clean_to_hourly_long(
    df_2024_raw,
    source_year=2024,
    has_passenger_type=False
)

station_hourly_2025_by_type = clean_to_hourly_long(
    df_2025_raw,
    source_year=2025,
    has_passenger_type=True
)

station_hourly_2025_total = (
    station_hourly_2025_by_type
    .groupby(
        [
            "service_date", "source_year",
            "line_name", "station_code", "station_name",
            "direction", "hour_bucket",
            "day_of_week", "is_weekday", "is_weekend",
            "month", "year_month"
        ],
        as_index=False
    )["passenger_count"]
    .sum()
)

station_hourly_2025_total["passenger_type"] = "all_passengers"
station_hourly_2025_total = station_hourly_2025_total[
    station_hourly_2024.columns
]

station_hourly_total = pd.concat(
    [station_hourly_2024, station_hourly_2025_total],
    ignore_index=True
)

# Aggregate validation 

In [36]:
typed_total = (
    station_hourly_2025_by_type
    .groupby(
        [
            "service_date", "line_name", "station_code",
            "direction", "hour_bucket"
        ],
        as_index=False
    )["passenger_count"]
    .sum()
)

aggregate_total = (
    station_hourly_2025_total
    .groupby(
        [
            "service_date", "line_name", "station_code",
            "direction", "hour_bucket"
        ],
        as_index=False
    )["passenger_count"]
    .sum()
)

validation = typed_total.merge(
    aggregate_total,
    on=[
        "service_date", "line_name", "station_code",
        "direction", "hour_bucket"
    ],
    suffixes=("_typed", "_aggregate")
)

assert (
    validation["passenger_count_typed"]
    == validation["passenger_count_aggregate"]
).all()

print("2025 passenger-type aggregation passed.")

2025 passenger-type aggregation passed.


# Create daily and station dimensions


In [37]:
station_daily = (
    station_hourly_total
    .groupby(
        [
            "service_date", "source_year",
            "line_name", "station_code", "station_name",
            "direction", "day_of_week",
            "is_weekday", "is_weekend", "month", "year_month"
        ],
        as_index=False
    )["passenger_count"]
    .sum()
)

station_daily_wide = (
    station_daily
    .pivot_table(
        index=[
            "service_date", "source_year",
            "line_name", "station_code", "station_name",
            "day_of_week", "is_weekday", "is_weekend",
            "month", "year_month"
        ],
        columns="direction",
        values="passenger_count",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
    .rename(columns={"board": "boardings", "alight": "alightings"})
)

station_daily_wide["total_activity"] = (
    station_daily_wide["boardings"]
    + station_daily_wide["alightings"]
)

station_daily_wide["net_boarding"] = (
    station_daily_wide["boardings"]
    - station_daily_wide["alightings"]
)

dim_station = (
    station_hourly_total[
        ["line_name", "station_code", "station_name"]
    ]
    .drop_duplicates()
    .sort_values(["line_name", "station_code"])
)

# Append the standardized 2024 and 2025 layers

In [38]:
station_hourly_total = pd.concat(
    [station_hourly_2024, station_hourly_2025_total],
    ignore_index=True
).sort_values(
    [
        "service_date", "line_name", "station_code",
        "direction", "hour_bucket"
    ]
).reset_index(drop=True)

print(station_hourly_total.shape)
print(station_hourly_total.head())
print(station_hourly_total.tail())


(5997760, 14)
  service_date  source_year line_name station_code station_name direction  \
0   2024-01-01         2024       1호선          150          서울역    alight   
1   2024-01-01         2024       1호선          150          서울역    alight   
2   2024-01-01         2024       1호선          150          서울역    alight   
3   2024-01-01         2024       1호선          150          서울역    alight   
4   2024-01-01         2024       1호선          150          서울역    alight   

   passenger_type hour_bucket  passenger_count day_of_week  is_weekday  \
0  all_passengers       06_07              867      Monday        True   
1  all_passengers       07_08              834      Monday        True   
2  all_passengers       08_09             1201      Monday        True   
3  all_passengers       09_10             1744      Monday        True   
4  all_passengers       10_11             1731      Monday        True   

   is_weekend  month year_month  
0       False      1    2024-01  
1       Fa

# Export outputs

In [ ]:
station_hourly_total.to_parquet(
    CLEANED_DIR / "station_hourly_total.parquet",
    index=False
)

station_hourly_2025_by_type.to_parquet(
    CLEANED_DIR / "station_hourly_passenger_type_2025.parquet",
    index=False
)

station_daily_wide.to_csv(
    CLEANED_DIR / "station_daily_wide.csv",
    index=False,
    encoding="utf-8-sig"
)

dim_station.to_csv(
    CLEANED_DIR / "dim_station.csv",
    index=False,
    encoding="utf-8-sig"
)